In [1]:
# Step 1 - Import libraries

import pandas as pd
import numpy as np
import os

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Step 2 - Load the working dataset

file_path = "../processed_data/water_quality_100k.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (100000, 14)


In [3]:
# Step 3 - Check columns

print("Columns:")
for i, col in enumerate(df.columns):
    print(i, ":", col)

Columns:
0 : Country
1 : Area
2 : Waterbody Type
3 : Date
4 : Ammonia (mg/l)
5 : Biochemical Oxygen Demand (mg/l)
6 : Dissolved Oxygen (mg/l)
7 : Orthophosphate (mg/l)
8 : pH (ph units)
9 : Temperature (cel)
10 : Nitrogen (mg/l)
11 : Nitrate (mg/l)
12 : CCME_Values
13 : CCME_WQI


In [4]:
# Step 4 - Check target variable

target = "Dissolved Oxygen (mg/l)"

print("Target:", target)
print("Missing target values:", df[target].isnull().sum())
print("Target mean:", df[target].mean())
print("Target median:", df[target].median())

Target: Dissolved Oxygen (mg/l)
Missing target values: 0
Target mean: 10.008087138942834
Target median: 10.2


In [5]:
# Step 5 - Convert Date to datetime

df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

print(df["Date"].dtype)
print("Invalid dates:", df["Date"].isnull().sum())

datetime64[us]
Invalid dates: 0


In [6]:
# Step 6 - Create Year feature

df["Year"] = df["Date"].dt.year

print(df[["Date", "Year"]].head())

        Date  Year
0 2006-10-31  2006
1 2006-06-01  2006
2 2013-08-07  2013
3 2005-06-07  2005
4 2014-05-14  2014


In [7]:
# Step 7 - Create Month feature

df["Month"] = df["Date"].dt.month

print(df[["Date", "Month"]].head())

        Date  Month
0 2006-10-31     10
1 2006-06-01      6
2 2013-08-07      8
3 2005-06-07      6
4 2014-05-14      5


In [8]:
# Step 7 - Create Month feature

df["Month"] = df["Date"].dt.month

print(df[["Date", "Month"]].head())

        Date  Month
0 2006-10-31     10
1 2006-06-01      6
2 2013-08-07      8
3 2005-06-07      6
4 2014-05-14      5


In [9]:
# Step 8 - Create Season feature

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["Season"] = df["Month"].apply(get_season)

print(df[["Date", "Month", "Season"]].head(10))

        Date  Month  Season
0 2006-10-31     10  Autumn
1 2006-06-01      6  Summer
2 2013-08-07      8  Summer
3 2005-06-07      6  Summer
4 2014-05-14      5  Spring
5 2015-01-14      1  Winter
6 2022-06-09      6  Summer
7 2000-09-06      9  Autumn
8 2003-06-05      6  Summer
9 2022-09-07      9  Autumn


In [10]:
print(df["Season"].value_counts())

Season
Autumn    26516
Summer    26118
Spring    24735
Winter    22631
Name: count, dtype: int64


In [11]:
# Step 9 - Check invalid negative values

concentration_cols = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Orthophosphate (mg/l)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)"
]

for col in concentration_cols:
    negative_count = (df[col] < 0).sum()
    print(f"{col}: {negative_count} negative values")

Ammonia (mg/l): 0 negative values
Biochemical Oxygen Demand (mg/l): 0 negative values
Dissolved Oxygen (mg/l): 0 negative values
Orthophosphate (mg/l): 1 negative values
Nitrogen (mg/l): 0 negative values
Nitrate (mg/l): 0 negative values


In [12]:
# Step 10 - Locate negative Orthophosphate

df.loc[
    df["Orthophosphate (mg/l)"] < 0,
    [
        "Country",
        "Area",
        "Waterbody Type",
        "Date",
        "Orthophosphate (mg/l)"
    ]
]

,Country,Area,Waterbody Type,Date,Orthophosphate (mg/l)
21416,Ireland,"Suir, Ballyshunnock",Lake,2007-04-30,-0.003


In [13]:
# Step 11 - Check extreme pH values

pH_invalid = (
    (df["pH (ph units)"] < 0) |
    (df["pH (ph units)"] > 14)
)

print("pH values outside 0-14:", pH_invalid.sum())

pH values outside 0-14: 21


In [14]:
df.loc[
    pH_invalid,
    [
        "Country",
        "Area",
        "Waterbody Type",
        "Date",
        "pH (ph units)"
    ]
].head(20)

,Country,Area,Waterbody Type,Date,pH (ph units)
910,Ireland,"Shannon Estuary South, Deel Estuary",Transitional,2015-06-09,24.60
3162,Ireland,"Ovoca-Vartry, Ballyglass",River,2019-04-29,19.50
10442,Ireland,"Shannon Estuary South, Limerick Dock",Transitional,2016-05-03,17.29
16438,Ireland,"Shannon Estuary South, Deel Estuary",Transitional,2023-03-07,24.60
16980,Ireland,"Ovoca-Vartry, Southwestern Irish Sea (HAs 11;12)",Coastal,2015-05-21,16.30
33420,Ireland,"Shannon Estuary South, Maigue Estuary",Transitional,2015-08-18,24.60
34360,Ireland,"Shannon Estuary South, MAHORE_020",River,2018-09-04,15.50
38312,Ireland,"Shannon Estuary South, Upper Shannon Estuary",Transitional,2019-11-05,24.30
39562,Ireland,"Shannon Estuary South, Limerick Dock",Transitional,2016-01-20,15.97
44389,Ireland,"Shannon Estuary South, Deel Estuary",Transitional,2016-06-07,24.00


In [15]:
# Step 12 - Check temperature extremes

print("Temperature < 0:", (df["Temperature (cel)"] < 0).sum())
print("Temperature > 50:", (df["Temperature (cel)"] > 50).sum())
print("Temperature > 60:", (df["Temperature (cel)"] > 60).sum())

Temperature < 0: 1
Temperature > 50: 203
Temperature > 60: 150


In [16]:
# Step 13 - Missing value check

missing = df.isnull().sum()

print(missing[missing > 0])

Series([], dtype: int64)


In [17]:
# Step 14 - Handle invalid pH values

pH_invalid = (
    (df["pH (ph units)"] < 0) |
    (df["pH (ph units)"] > 14)
)

print("Invalid pH values before:", pH_invalid.sum())

df.loc[pH_invalid, "pH (ph units)"] = np.nan

print("Missing pH values after:", df["pH (ph units)"].isnull().sum())

Invalid pH values before: 21
Missing pH values after: 21


In [18]:
# Step 15 - Handle negative Orthophosphate

negative_orthophosphate = df["Orthophosphate (mg/l)"] < 0

print(
    "Negative Orthophosphate before:",
    negative_orthophosphate.sum()
)

df.loc[
    negative_orthophosphate,
    "Orthophosphate (mg/l)"
] = np.nan

print(
    "Missing Orthophosphate after:",
    df["Orthophosphate (mg/l)"].isnull().sum()
)

Negative Orthophosphate before: 1
Missing Orthophosphate after: 1


In [19]:
# Step 16 - Fill missing numerical values with median

numeric_features = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Temperature (cel)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)"
]

for col in numeric_features:
    df[col] = df[col].fillna(df[col].median())

print("Remaining missing values:")
print(df[numeric_features].isnull().sum())

Remaining missing values:
Ammonia (mg/l)                      0
Biochemical Oxygen Demand (mg/l)    0
Orthophosphate (mg/l)               0
pH (ph units)                       0
Temperature (cel)                   0
Nitrogen (mg/l)                     0
Nitrate (mg/l)                      0
dtype: int64


In [20]:
# Step 17 - Investigate high temperature observations

high_temp = df["Temperature (cel)"] > 50

print("Temperature > 50:", high_temp.sum())

print("\nHigh temperature observations by country:")
print(
    df.loc[
        high_temp
    ].groupby("Country").size().sort_values(ascending=False)
)

print("\nHigh temperature observations by waterbody:")
print(
    df.loc[
        high_temp
    ].groupby("Waterbody Type").size().sort_values(ascending=False)
)

Temperature > 50: 203

High temperature observations by country:
Country
USA        199
Ireland      3
England      1
dtype: int64

High temperature observations by waterbody:
Waterbody Type
River           200
Transitional      3
dtype: int64


In [21]:
# Step 18 - Inspect high temperature records

df.loc[
    high_temp,
    [
        "Country",
        "Area",
        "Waterbody Type",
        "Date",
        "Temperature (cel)",
        "Dissolved Oxygen (mg/l)"
    ]
].head(30)

,Country,Area,Waterbody Type,Date,Temperature (cel),Dissolved Oxygen (mg/l)
77,USA,Old River at Sand Mound Slough,River,1973-07-16,70.0,9.870000
171,USA,SACRAMENTO R A HAMILTON CITY,River,1955-08-18,58.0,9.700000
780,USA,MURRY C NR SAN ANDREAS,River,1958-06-24,70.0,8.800000
910,Ireland,"Shannon Estuary South, Deel Estuary",Transitional,2015-06-09,51.3,4.972500
1886,USA,MOKELUMNE R A LANCHA PLANA,River,1961-03-01,52.0,10.900000
2175,USA,D12A - San Joaquin River @ Antioch,River,1955-05-19,69.0,8.800000
2906,USA,D12A - San Joaquin River @ Antioch,River,1951-05-15,61.7,9.000000
3863,USA,BUTTE C OPP COLUSA BP NR COLUSA,River,1959-07-20,78.0,9.870000
3931,USA,Sacramento River @ Mallard Island - D10A,River,1978-05-08,62.0,9.870000
4127,USA,INDIAN SLU NR BRENTWOOD,River,1962-03-08,60.0,9.200000


In [22]:
# Step 19 - Create temperature extreme indicator

df["Temperature_Extreme"] = (
    (df["Temperature (cel)"] < 0) |
    (df["Temperature (cel)"] > 50)
).astype(int)

print(
    "Extreme temperature observations:",
    df["Temperature_Extreme"].sum()
)

print(
    df["Temperature_Extreme"].value_counts()
)

Extreme temperature observations: 204
Temperature_Extreme
0    99796
1      204
Name: count, dtype: int64


In [23]:
# Step 20 - Final numerical data quality check

numeric_check_cols = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Temperature (cel)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)"
]

quality_check = pd.DataFrame({
    "Missing": df[numeric_check_cols].isnull().sum(),
    "Minimum": df[numeric_check_cols].min(),
    "Maximum": df[numeric_check_cols].max()
})

quality_check

,Missing,Minimum,Maximum
Ammonia (mg/l),0,0.0,197.0
Biochemical Oxygen Demand (mg/l),0,0.0,255.0
Dissolved Oxygen (mg/l),0,0.0,20.0
Orthophosphate (mg/l),0,0.0,100.0
pH (ph units),0,0.0,13.3
Temperature (cel),0,-1.0,90.0
Nitrogen (mg/l),0,0.0,46.0
Nitrate (mg/l),0,0.0,155.0


In [24]:
# Step 21 - Define features and target

feature_cols = [
    "Country",
    "Waterbody Type",
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Temperature (cel)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)",
    "Year",
    "Month",
    "Season",
    "Temperature_Extreme"
]

target_col = "Dissolved Oxygen (mg/l)"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Features:", X.shape)
print("Target:", y.shape)

Features: (100000, 13)
Target: (100000,)


In [25]:
# Step 22 - Check feature data types

print(X.dtypes)

Country                                 str
Waterbody Type                          str
Ammonia (mg/l)                      float64
Biochemical Oxygen Demand (mg/l)    float64
Orthophosphate (mg/l)               float64
pH (ph units)                       float64
Temperature (cel)                   float64
Nitrogen (mg/l)                     float64
Nitrate (mg/l)                      float64
Year                                  int32
Month                                 int32
Season                                  str
Temperature_Extreme                   int64
dtype: object


In [26]:
# Step 23 - Check categorical features

categorical_features = [
    "Country",
    "Waterbody Type",
    "Season"
]

for col in categorical_features:
    print(f"\n{col}")
    print("Unique values:", X[col].nunique())
    print(X[col].unique())


Country
Unique values: 5
<ArrowStringArray>
['England', 'Ireland', 'USA', 'China', 'Canada']
Length: 5, dtype: str

Waterbody Type
Unique values: 12
<ArrowStringArray>
[    'Effluent',        'River',    'Estuarine',    'Sea Water',
         'Lake',          'Bay',       'Sewage',       'Marine',
     'Drainage',        'Canal', 'Transitional',      'Coastal']
Length: 12, dtype: str

Season
Unique values: 4
<ArrowStringArray>
['Autumn', 'Summer', 'Spring', 'Winter']
Length: 4, dtype: str


In [27]:
# Step 24 - Check numerical features

numerical_features = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Temperature (cel)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)",
    "Year",
    "Month",
    "Temperature_Extreme"
]

print(X[numerical_features].describe().T)

                                     count         mean        std     min  \
Ammonia (mg/l)                    100000.0     1.141159   5.491133     0.0   
Biochemical Oxygen Demand (mg/l)  100000.0     4.832689  16.149127     0.0   
Orthophosphate (mg/l)             100000.0     0.708422   2.069572     0.0   
pH (ph units)                     100000.0     7.732821   0.460927     0.0   
Temperature (cel)                 100000.0    11.819261   4.995563    -1.0   
Nitrogen (mg/l)                   100000.0     5.207477   6.222716     0.0   
Nitrate (mg/l)                    100000.0     4.792220   6.187033     0.0   
Year                              100000.0  2006.878870  12.494377  1940.0   
Month                             100000.0     6.461050   3.384145     1.0   
Temperature_Extreme               100000.0     0.002040   0.045120     0.0   

                                      25%       50%          75%     max  
Ammonia (mg/l)                       0.03     0.054     0.316158  

In [28]:
# Step 25 - Check observations available for each year

year_counts = (
    df["Year"]
    .value_counts()
    .sort_index()
)

print(year_counts)

Year
1940       2
1941       8
1942      10
1943       4
1944       8
        ... 
2019    3068
2020    1743
2021    2574
2022    2755
2023    1897
Name: count, Length: 84, dtype: int64


In [29]:
# Step 26 - Check observations in proposed periods

print("Training period (<= 2018):",
      (df["Year"] <= 2018).sum())

print("Validation period (2019-2021):",
      ((df["Year"] >= 2019) & (df["Year"] <= 2021)).sum())

print("Test period (>= 2022):",
      (df["Year"] >= 2022).sum())

Training period (<= 2018): 87963
Validation period (2019-2021): 7385
Test period (>= 2022): 4652


In [30]:
# Step 27 - Create time-based train, validation and test sets

train_mask = df["Year"] <= 2018

validation_mask = (
    (df["Year"] >= 2019) &
    (df["Year"] <= 2021)
)

test_mask = df["Year"] >= 2022

train_df = df.loc[train_mask].copy()
validation_df = df.loc[validation_mask].copy()
test_df = df.loc[test_mask].copy()

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Training shape: (87963, 18)
Validation shape: (7385, 18)
Test shape: (4652, 18)


In [31]:
# Step 28 - Separate features and target

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_validation = validation_df[feature_cols].copy()
y_validation = validation_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (87963, 13)
y_train: (87963,)
X_validation: (7385, 13)
y_validation: (7385,)
X_test: (4652, 13)
y_test: (4652,)


In [32]:
# Step 29 - Compare target distribution across splits

target_summary = pd.DataFrame({
    "Train": y_train.describe(),
    "Validation": y_validation.describe(),
    "Test": y_test.describe()
})

target_summary

,Train,Validation,Test
count,87963.000000,7385.000000,4652.000000
mean,10.035294,9.833187,9.771287
std,1.800573,2.208385,2.199568
min,0.000000,0.174000,0.101000
25%,9.870000,9.600000,9.376000
50%,10.200000,10.200000,10.200000
75%,10.900000,11.300000,11.200000
max,20.000000,19.900000,18.750000


In [33]:
# Step 30 - Check categorical values in each split

for col in ["Country", "Waterbody Type", "Season"]:
    print("\n==============================")
    print(col)
    
    print("Train:")
    print(X_train[col].unique())
    
    print("Validation:")
    print(X_validation[col].unique())
    
    print("Test:")
    print(X_test[col].unique())


Country
Train:
<ArrowStringArray>
['England', 'USA', 'Ireland', 'China', 'Canada']
Length: 5, dtype: str
Validation:
<ArrowStringArray>
['England', 'USA', 'Ireland', 'Canada']
Length: 4, dtype: str
Test:
<ArrowStringArray>
['England', 'Ireland', 'USA']
Length: 3, dtype: str

Waterbody Type
Train:
<ArrowStringArray>
[    'Effluent',        'River',    'Estuarine',    'Sea Water',
         'Lake',          'Bay',       'Sewage',       'Marine',
     'Drainage',        'Canal', 'Transitional',      'Coastal']
Length: 12, dtype: str
Validation:
<ArrowStringArray>
[    'Effluent',        'River',         'Lake',       'Marine',
     'Drainage',    'Sea Water', 'Transitional',    'Estuarine',
      'Coastal',       'Sewage',        'Canal']
Length: 11, dtype: str
Test:
<ArrowStringArray>
[       'River',     'Effluent',         'Lake',       'Marine',
    'Estuarine',      'Coastal', 'Transitional',        'Canal',
    'Sea Water',     'Drainage',       'Sewage']
Length: 11, dtype: str

Sea

In [34]:
# Step 31 - Import preprocessing tools

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [35]:
# Step 32 - Define preprocessing columns

categorical_features = [
    "Country",
    "Waterbody Type",
    "Season"
]

numerical_features = [
    "Ammonia (mg/l)",
    "Biochemical Oxygen Demand (mg/l)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Temperature (cel)",
    "Nitrogen (mg/l)",
    "Nitrate (mg/l)",
    "Year",
    "Month",
    "Temperature_Extreme"
]

In [36]:
# Step 33 - Create preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully")

Preprocessor created successfully


In [37]:
# Step 34 - Fit preprocessing on training data only

X_train_processed = preprocessor.fit_transform(X_train)

X_validation_processed = preprocessor.transform(X_validation)

X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Validation processed shape:", X_validation_processed.shape)
print("Test processed shape:", X_test_processed.shape)

Training processed shape: (87963, 31)
Validation processed shape: (7385, 31)
Test processed shape: (4652, 31)


In [38]:
# Step 35 - Check processed data

print("Training:")
print(X_train_processed.shape)

print("\nValidation:")
print(X_validation_processed.shape)

print("\nTest:")
print(X_test_processed.shape)

print("\nData type:")
print(X_train_processed.dtype)

Training:
(87963, 31)

Validation:
(7385, 31)

Test:
(4652, 31)

Data type:
float64


In [39]:
# Step 36 - Save processed datasets

import os

os.makedirs("../processed_data", exist_ok=True)

np.save(
    "../processed_data/X_train_processed.npy",
    X_train_processed
)

np.save(
    "../processed_data/X_validation_processed.npy",
    X_validation_processed
)

np.save(
    "../processed_data/X_test_processed.npy",
    X_test_processed
)

y_train.to_csv(
    "../processed_data/y_train.csv",
    index=False
)

y_validation.to_csv(
    "../processed_data/y_validation.csv",
    index=False
)

y_test.to_csv(
    "../processed_data/y_test.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


In [40]:
# Step 37 - Save preprocessing pipeline

import joblib

joblib.dump(
    preprocessor,
    "../models/water_quality_preprocessor.joblib"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.
